# CareSignal ML Lab — SIH 26094

Demo the full AI stack **without** the web UI: trained text model, voice emotion proxy, distress fusion, escalation logistic.

### Training data used in this build
| Dataset | Role | Rows (approx) |
|---|---|---|
| GoEmotions | fine emotion / polarity | 207k prepared |
| SentiMix Hinglish | code-mixed sentiment | 17k |
| IndicSentiment | Indic + English polarity | 27k |
| Kaggle emotion+sentiment | 6 emotions + pos/neg | 422k + 3k |
| Domain bootstrap | justice / victim phrases | 40 |
| Synthetic longitudinal | escalation logistic | 2.4k |

**Text model:** `tfidf-logistic-v3-kaggle` — hold-out accuracy ~84%, macro F1 ~0.80  
**Voice model:** `voice-linear-v1` acoustic proxy  
**Escalation:** `synthetic-logistic-v1`

> Prototype only — not clinical validation.


In [ ]:
# Deps (skip quietly if already installed)
import sys, subprocess
pkgs = ['numpy', 'scipy', 'scikit-learn', 'matplotlib']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])
print('deps ok')


## 1) Locate repo + load live CareSignal AI


In [ ]:
from pathlib import Path
import sys, os, json, math, io, wave, struct, re
from datetime import datetime, timezone
import numpy as np
from scipy.fft import dct
import matplotlib.pyplot as plt

CANDIDATES = [
    Path('/content/CareSignal'),
    Path.cwd(),
    Path.cwd().parent,
]
REPO_ROOT = next((p for p in CANDIDATES if (p / 'backend' / 'app' / 'ai.py').exists()), None)
assert REPO_ROOT, 'Upload/clone CareSignal so backend/app/ai.py is visible'
sys.path.insert(0, str(REPO_ROOT / 'backend'))
os.environ.setdefault('AI_MODE', 'demo')

# Force reload so notebook picks up latest models
import importlib
import app.config as config
import app.ai as ai
importlib.reload(config)
importlib.reload(ai)
ai._TEXT_MODEL = None
ai._VOICE_MODEL = None

from app.ai import analyze_text, audio_features, calculate, predict, category
from app.config import WEIGHTS, THRESHOLDS, MODEL_PATH

print('REPO_ROOT =', REPO_ROOT)
print('MODEL_PATH =', MODEL_PATH)
for name in ['text_sentiment.json', 'text_sentiment_metrics.json', 'voice_emotion.json', 'voice_emotion_metrics.json', 'logistic.json', 'metrics.json']:
    p = MODEL_PATH / name
    print(('OK ' if p.exists() else 'MISSING '), name)


## 2) Dataset inventory (processed JSONL on disk)


In [ ]:
proc = REPO_ROOT / 'data' / 'processed'
raw = REPO_ROOT / 'data' / 'raw'
inventory = []
for p in sorted(proc.glob('*.jsonl')):
    n = sum(1 for _ in p.open(encoding='utf-8'))
    inventory.append({'file': p.name, 'rows': n, 'MB': round(p.stat().st_size/1e6, 2)})
print('Processed:')
for row in inventory:
    print(f"  {row['file']:28} {row['rows']:>8} rows  {row['MB']} MB")

print('\nRaw folders present:')
for d in sorted(raw.iterdir()) if raw.exists() else []:
    if d.is_dir():
        print(' ', d.name)

# metrics
text_m = json.loads((MODEL_PATH/'text_sentiment_metrics.json').read_text()) if (MODEL_PATH/'text_sentiment_metrics.json').exists() else {}
voice_m = json.loads((MODEL_PATH/'voice_emotion_metrics.json').read_text()) if (MODEL_PATH/'voice_emotion_metrics.json').exists() else {}
esc_m = json.loads((MODEL_PATH/'metrics.json').read_text()) if (MODEL_PATH/'metrics.json').exists() else {}
print('\nText model:', text_m.get('version'), 'macro_f1=', text_m.get('macro_f1_holdout'), 'n_train=', text_m.get('n_train'))
print('Voice model:', voice_m.get('version'), 'macro_f1=', voice_m.get('macro_f1_holdout'))
print('Escalation:', esc_m.get('version'), 'acc=', esc_m.get('accuracy'), 'auc=', esc_m.get('roc_auc'))


## 3) Sentiment + Emotion AI (live model)


In [ ]:
samples = {
  'calm_en': 'I am feeling better and feel safe today. Doing well overall.',
  'distress_en': 'I am scared. Someone threatened my family. I have not slept and feel hopeless.',
  'hindi': 'मुझे डर लग रहा है। धमकी मिली है और नींद नहीं आ रही।',
  'hinglish': 'Bahut darr lag raha hai, dhamki mili, neend nahi aa rahi, akela feel kar raha hoon.',
  'joy': 'I feel joyful and grateful today.',
  'anger': 'I am so angry about this injustice and delay.',
  'negated': 'I am not afraid and there is no threat to my family.',
}
rows = []
for name, text in samples.items():
    r = analyze_text(text)
    active = [k for k,v in r['signals'].items() if v]
    rows.append((name, r['sentiment']['label'], r['sentiment']['score'], r['emotions'], active, r['method']))
    print(f"\n[{name}] {r['sentiment']}  method={r['method']}")
    print(' emotions:', {k:v for k,v in r['emotions'].items() if v})
    print(' signals:', active)

labels = [r[0] for r in rows]
# signed score: +neg intensity, -pos
scores = []
for r in rows:
    if r[1]=='negative': scores.append(r[2])
    elif r[1]=='positive': scores.append(-r[2])
    else: scores.append(0)
colors = ['#e76f51' if s>0 else '#2a9d8f' if s<0 else '#8d99ae' for s in scores]
plt.figure(figsize=(9,3.2))
plt.bar(labels, scores, color=colors)
plt.axhline(0, color='#333', lw=.8)
plt.title('Sentiment intensity (negative > 0, positive < 0)')
plt.xticks(rotation=25, ha='right'); plt.tight_layout(); plt.show()

# emotion heatmap-ish
emo_keys = ['fear','sadness','anger','joy','neutral']
mat = np.array([[r[3].get(k,0) for k in emo_keys] for r in rows])
fig, ax = plt.subplots(figsize=(7,3.5))
im = ax.imshow(mat, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
ax.set_xticks(range(len(emo_keys))); ax.set_xticklabels(emo_keys)
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
ax.set_title('Emotion intensities')
fig.colorbar(im, ax=ax, fraction=0.046); plt.tight_layout(); plt.show()


## 4) Voice stress / emotion proxy


In [ ]:
def make_synth_wav(seconds=2.0, rate=16000, stressed=True):
    n = int(rate * seconds)
    buf = io.BytesIO()
    with wave.open(buf, 'wb') as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(rate)
        frames = bytearray()
        for i in range(n):
            t = i / rate
            if stressed:
                env = 0.35 if (i // 1600) % 2 == 0 else 0.05
                f0 = 180 + 40 * math.sin(2 * math.pi * 3 * t)
            else:
                env = 0.2
                f0 = 140 + 5 * math.sin(2 * math.pi * 1 * t)
            sample = env * (0.6 * math.sin(2 * math.pi * f0 * t) + 0.25 * math.sin(2 * math.pi * 2 * f0 * t))
            frames += struct.pack('<h', int(max(-1, min(1, sample)) * 30000))
        w.writeframes(frames)
    return buf.getvalue()

upload = Path('/content/sample.wav')
local_sample = REPO_ROOT / 'data' / 'raw' / 'sample.wav'
raw = (upload if upload.exists() else local_sample).read_bytes() if (upload.exists() or local_sample.exists()) else None

results = {}
for name, stressed in [('stressed', True), ('calm', False)]:
    audio = raw if (name=='stressed' and raw) else make_synth_wav(stressed=stressed)
    vf = audio_features(audio, transcript='I am scared and cannot sleep' if stressed else 'I feel okay today')
    results[name] = vf
    print(f"\n=== {name} ===")
    print({k: vf.get(k) for k in ['duration','energy','pause_ratio','pitch_mean','pitch_variability','emotion','stress_score','emotion_method']})

fig, axes = plt.subplots(1,2, figsize=(10,2.5))
for ax, (name, vf) in zip(axes, results.items()):
    # reconstruct waveform for plot from synth only
    audio = make_synth_wav(stressed=(name=='stressed'))
    with wave.open(io.BytesIO(audio), 'rb') as w:
        sig = np.frombuffer(w.readframes(w.getnframes()), dtype='<i2').astype(float)
        rate = w.getframerate()
    ax.plot(np.linspace(0, len(sig)/rate, len(sig)), sig, lw=.5)
    ax.set_title(f"{name}: emotion={vf.get('emotion')} stress={vf.get('stress_score')}")
plt.tight_layout(); plt.show()


## 5) Dynamic Distress Score + Predictive Escalation


In [ ]:
high = calculate(
    {'feeling':3,'fear':3,'daily':2,'avoidance':2,'legal':1,'sleep':1,'threat':True,'safe':False},
    'I am scared. Brother got a threat. Court hearing next week. Have not slept.',
    'en',
    [{'at':'2026-08-01T10:00:00','score':35},{'at':'2026-08-15T10:00:00','score':42},{'at':'2026-09-01T10:00:00','score':55}],
    {'threat':True,'court_event':True,'missed_checkins':2,'investigation_delay':1,'compensation_delay':0},
    datetime.now(timezone.utc),
    voice={**results['stressed'], 'baseline_deviation': 0.35},
)
calm = calculate(
    {'feeling':0,'fear':0,'daily':0,'avoidance':0,'legal':0,'sleep':3,'threat':False,'safe':True},
    'Feeling better and feel safe. Doing well.',
    'en',
    [{'at':'2026-08-01T10:00:00','score':40},{'at':'2026-08-15T10:00:00','score':30},{'at':'2026-09-01T10:00:00','score':22}],
    {'threat':False,'court_event':False,'missed_checkins':0},
    datetime.now(timezone.utc),
    voice=results['calm'],
)
for label, c in [('HIGH RISK', high), ('CALM', calm)]:
    print(f"\n=== {label} ===")
    print('scores:', c['scores'], 'priority:', c['priority'])
    print('prediction:', c['prediction'])
    print('sentiment:', c['sentiment'])
    print('top factors:', sorted(c['explanation'], key=lambda e: -e['impact'])[:4])

fig, ax = plt.subplots(figsize=(6.5,3.2))
x = np.arange(3); w = 0.35
ax.bar(x-w/2, [high['scores']['distress'], high['scores']['safety_risk'], high['scores']['escalation_risk']], w, label='High', color='#e76f51')
ax.bar(x+w/2, [calm['scores']['distress'], calm['scores']['safety_risk'], calm['scores']['escalation_risk']], w, label='Calm', color='#2a9d8f')
ax.set_xticks(x); ax.set_xticklabels(['Distress','Safety','Escalation']); ax.set_ylim(0,100)
ax.legend(); ax.set_title('Score comparison (live CareSignal pipeline)'); plt.tight_layout(); plt.show()


## 6) Model metrics dashboards


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

# text classes F1 not stored per-class in metrics json — show summary bars
if text_m:
    axes[0].bar(['macro_f1'], [text_m.get('macro_f1_holdout',0)], color='#457b9d')
    axes[0].set_ylim(0,1); axes[0].set_title(f"Text {text_m.get('version','')}")
    axes[0].text(0, text_m.get('macro_f1_holdout',0)+0.02, f"{text_m.get('macro_f1_holdout',0):.2f}", ha='center')
if voice_m:
    axes[1].bar(['macro_f1'], [voice_m.get('macro_f1_holdout',0)], color='#e9c46a')
    axes[1].set_ylim(0,1); axes[1].set_title(f"Voice {voice_m.get('version','')}")
if esc_m:
    keys = ['accuracy','precision','recall','f1','roc_auc']
    vals = [esc_m.get(k,0) for k in keys]
    axes[2].bar(keys, vals, color='#e76f51')
    axes[2].set_ylim(0,1); axes[2].set_title('Escalation logistic')
    axes[2].tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

if esc_m.get('feature_importance'):
    names = [f['name'] for f in esc_m['feature_importance']]
    vals = [abs(f['value']) for f in esc_m['feature_importance']]
    plt.figure(figsize=(8,3.2)); plt.barh(names, vals); plt.title('|coef| escalation features'); plt.tight_layout(); plt.show()

print('Text limitations:', text_m.get('limitations'))
print('Escalation limitations:', esc_m.get('limitations'))


## 7) Try your own text


In [ ]:
YOUR_TEXT = "Mujhe bahut darr lag raha hai. Dhamki mili hai aur neend nahi aa rahi."
r = analyze_text(YOUR_TEXT)
print(json.dumps({k:r[k] for k in ['sentiment','emotions','method','signals']}, ensure_ascii=False, indent=2))


## Judge talking points

| Component | Evidence in this notebook |
|---|---|
| Multilingual NLP | EN / Hindi / Hinglish samples + SentiMix + IndicSentiment training |
| Emotion AI | TF-IDF multi-class head (fear/sadness/anger/joy/…) + domain lexicon |
| Voice stress | Acoustic features + trained/heuristic emotion proxy |
| Distress score | Weighted fusion with explanations |
| Escalation prediction | Synthetic logistic with held-out metrics |

Keep this notebook as the fallback demo if the web stack misbehaves.
